# 01 — A First Binary Decoder (pos_val vs neg_val)

**Purpose:** Build your first brain decoder — a support vector machine (SVM) that
tries to tell, from a single-trial beta map, whether the trial was a
**positive-valence** or **negative-valence** emotional experience.

This notebook leans on Nilearn's high-level `Decoder` object, which bundles
masking, feature selection, the classifier, and cross-validation into one tool.
We build it up slowly so you understand *every* piece, rather than treating it as
a black box.

> Reference: this notebook mirrors the structure of Nilearn's decoding tutorials
> (the Haxby "Decoding with ANOVA + SVM" example). If you want a second
> explanation of any step, that example is the canonical source.

---

## Notebook overview

1. Load `decoder_posval_vs_negval.csv`.
2. Turn the table into the three things a decoder needs: `imgs` (the images),
   `y` (the labels), and `groups` (the run identity for cross-validation).
3. Build a Nilearn `Decoder` with a brain mask and ANOVA feature selection.
4. Run leave-one-run-out cross-validation.
5. Read the scores against the correct chance level (50%).

You will **not** fully implement the decoder here. The `TODO`s leave the key
choices — the mask, the screening percentile, the fit call, the score
extraction — for you to write.

## What you are learning

- What **classification** is, in one sentence.
- How beta maps become **samples** and voxels become **features**.
- Why decoding needs a **mask**.
- What **ANOVA feature selection** does and why it must live *inside*
  cross-validation.
- What **data leakage** is and how it fakes good results.
- What **chance level** is and how to compare against it.
- A working intuition for how a linear **SVM** separates two classes.

## Why this matters scientifically

A univariate contrast (notebook 04 of the beta series) asks "does this *one*
voxel respond differently to positive vs negative valence?" A decoder asks a
richer question: "is there *any pattern across many voxels* that distinguishes
the two?" Above-chance decoding is evidence that valence is represented in the
*multivariate* structure of the BOLD response, even when no single voxel is
individually convincing. That is the core idea of MVPA (multi-voxel pattern
analysis).

---
## Section 1 — What "Classification" Means

**Classification** = learning a rule that maps an input vector to one of a fixed
set of categories. Training shows the algorithm many `(X_row, y_label)` pairs; it
adjusts an internal boundary so that, for new unseen rows, it predicts the right
label.

In our case:

- input vector `X_row` = one trial's voxel pattern (tens of thousands of numbers)
- category `y_label` = `"pos_val"` or `"neg_val"`

The decoder never sees brain anatomy or your hypothesis — only numbers and the
labels you attached. Everything scientific about the result comes from *how you
set up `X`, `y`, and the validation*. That is why we spend most of this notebook
on setup and only one line on the actual fit.

---
## Section 2 — Configuration and Load the Table

Load the decoder table you built in notebook 00. From it you will derive three
aligned objects, all by iterating the **same DataFrame in the same order**:

- `imgs`   — a list of image paths (or loaded images) → becomes `X`
- `y`      — the `label` column → the answers
- `groups` — the `groups` column → run identity for cross-validation

Reading the table:

```python
table = pd.read_csv(tables_dir / "decoder_posval_vs_negval.csv")
```

**TODO:**
- [ ] Set `subject` and build `tables_dir`.
- [ ] Read `decoder_posval_vs_negval.csv` into `table`.
- [ ] Print `table.shape` (expect `(32, 7)`) and `table["label"].value_counts()`.

In [6]:
from pathlib import Path
import pandas as pd

subject = "sub-097"

project_dir  = Path(r"C:\ManzaRotation")
decoding_dir = project_dir / "Analysis" / "outputs" / subject / "decoding"
tables_dir   = decoding_dir / "tables"

# TODO: read decoder_posval_vs_negval.csv into `table`
table = pd.read_csv(tables_dir / f"{subject}_posval_negval_catalog.csv")
# TODO: print table.shape and table["label"].value_counts()
print(table.shape)
print(table["label"].value_counts)

(32, 7)
<bound method IndexOpsMixin.value_counts of 0     neg_val
1     neg_val
2     neg_val
3     neg_val
4     neg_val
5     neg_val
6     neg_val
7     neg_val
8     pos_val
9     pos_val
10    pos_val
11    pos_val
12    pos_val
13    pos_val
14    pos_val
15    pos_val
16    neg_val
17    neg_val
18    neg_val
19    neg_val
20    neg_val
21    neg_val
22    neg_val
23    neg_val
24    pos_val
25    pos_val
26    pos_val
27    pos_val
28    pos_val
29    pos_val
30    pos_val
31    pos_val
Name: label, dtype: str>


---
## Section 3 — Build `imgs`, `y`, and `groups`

The Nilearn `Decoder` can take a **list of image file paths** directly — it will
load and mask them for you. So `imgs` can be as simple as the `path` column
turned into a list.

```python
imgs   = table["path"].tolist()      # list of 32 file paths -> X
y      = table["label"].tolist()     # list of 32 labels      -> answers
groups = table["groups"].tolist()    # list of 32 run names    -> CV folds
```

**Why `.tolist()`?** Nilearn and scikit-learn happily accept plain Python lists or
NumPy arrays; converting once now avoids pandas-index surprises later. The
critical invariant is **alignment**: `imgs[i]`, `y[i]`, and `groups[i]` must all
describe the same trial. Because they come from the same rows in the same order,
they are aligned by construction — do not sort or filter one without the others.

**TODO:**
- [ ] Build `imgs`, `y`, and `groups` from the three columns.
- [ ] Assert all three have the same length (`len(imgs) == len(y) == len(groups)`).
- [ ] Print the first 3 entries of each and eyeball that row 0 lines up.

In [16]:
# TODO: imgs   = table["path"]...
imgs = table["path"].tolist()
# TODO: y      = table["label"]...
y= table["label"].tolist()
# TODO: groups = table["groups"]...
groups = table["groups"]

# TODO: assert len(imgs) == len(y) == len(groups)
assert len(imgs)==len(y)==len(groups)
# TODO: print the first few of each to confirm alignment
check = pd.DataFrame({
    "imgs": imgs,
    "y": y,
    "groups": groups
})
check.head

<bound method NDFrame.head of                                                  imgs        y     groups
0   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
1   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
2   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
3   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
4   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
5   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
6   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
7   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_val  modulate1
8   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  pos_val  modulate1
9   C:\ManzaRotation\Analysis\outputs\sub-097\beta...  pos_val  modulate1
10  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  pos_val  modulate1
11  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  pos_val  modulate1
12  C:\M

---
## Section 4 — Why We Need a Mask

A beta map is a 3-D box of voxels, but a lot of that box is *outside the brain*
(skull, air, ventricic edges). Those voxels are noise. Two reasons a **mask**
matters:

1. **It defines the features.** The mask selects which voxels become columns of
   `X`. Out-of-brain voxels carry no signal but would add thousands of noisy
   features, making the decoder slower and easier to overfit.
2. **It fixes the feature space.** Every trial is masked with the *same* mask, so
   feature *j* means "the same voxel" across all samples. Without a common mask,
   the columns of `X` would not be comparable.

You already have a brain mask from the beta-series work:

```
C:\ManzaRotation\Derivatives\sub-097\func\
    sub-097_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz
```

A whole-brain mask is the simplest choice and the right one for a first pass.
Later you might swap in an anatomical ROI (e.g. an emotion-related region) to ask
a more targeted question — but that is a different scientific claim, so start
whole-brain.

**TODO:**
- [ ] Define `mask_path` pointing at the brain mask above.
- [ ] Confirm it exists with `mask_path.exists()`.

In [17]:
# TODO: point mask_path at the sub-097 brain mask in Derivatives/.../func
func_deriv_dir = project_dir / "Derivatives" / subject / "func"
mask_path = func_deriv_dir / f"{subject}_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz"

# TODO: print mask_path.exists()
print(mask_path.exists())

True


---
## Section 5 — ANOVA Feature Selection (and Why It Must Be *Inside* CV)

Even inside the brain mask you have ~50,000 voxels but only 32 samples. With far
more features than samples, a classifier can find a boundary that *perfectly*
fits the training data by exploiting noise — it **overfits**. Feature selection
fights this by keeping only the voxels that look most informative.

**ANOVA feature selection** scores each voxel by how differently its values
distribute across the two classes (an F-test), then keeps the top *k%* —
controlled by Nilearn's `screening_percentile`. A smaller percentile keeps fewer,
more selective voxels.

Here is the subtle, crucial part — **the rule you must never break:**

> Feature selection has to be re-computed on the **training data of each CV fold
> only**, never on the full dataset.

Why? Choosing voxels using *all* the data — including the trials you will later
test on — lets information from the test set leak into training. The decoder then
looks better than it really is. This is **data leakage**, and it is the most
common way fMRI decoding results turn out to be illusory.

The good news: **Nilearn's `Decoder` does this correctly for you.** When you pass
`screening_percentile`, the ANOVA is refit inside every fold automatically. You
get leak-free feature selection *for free* — as long as you let the `Decoder`
run the cross-validation, rather than selecting features by hand beforehand.

**TODO (no code — answer below):**
- [ ] In one sentence, what is data leakage?
- [ ] Why does selecting voxels on the full dataset count as leakage?

*Your answers:*

> (write here)

---
## Section 6 — Chance Level and the SVM Intuition

**Chance level.** This is a *balanced binary* problem (16 vs 16), so a decoder
that guesses randomly scores **50%**. Write that number down now. A result is
only interesting relative to chance — "70% accuracy" means nothing until you know
the bar was 50%.

**SVM intuition.** A linear support vector machine looks for the flat boundary (a
hyperplane) that separates the two classes with the widest possible *margin* — the
biggest gap between the boundary and the nearest trials of each class. The trials
sitting right on the margin are the "support vectors"; they alone define the
boundary. A wide margin tends to generalise better to new trials. For
high-dimensional fMRI patterns, a *linear* SVM is the standard, well-behaved
default — don't reach for fancier kernels first.

---
## Section 7 — Build the Nilearn `Decoder`

`nilearn.decoding.Decoder` wraps the whole pipeline: masking → ANOVA screening →
SVM → cross-validation. The arguments you care about now:

| argument | meaning |
|---|---|
| `estimator` | which classifier; `"svc"` is a linear SVM |
| `mask` | the brain mask image/path that defines the features |
| `standardize` | z-score each voxel before fitting (recommended) |
| `screening_percentile` | the ANOVA % of voxels to keep (refit per fold) |
| `scoring` | metric to report, e.g. `"accuracy"` |
| `cv` | the cross-validation splitter |

For cross-validation, use **leave-one-run-out** via scikit-learn's
`LeaveOneGroupOut`, which respects the `groups` array:

```python
from sklearn.model_selection import LeaveOneGroupOut
cv = LeaveOneGroupOut()
```

With two runs this yields exactly two folds: train on `modulate1` / test on
`modulate2`, and the reverse. Skeleton:

```python
from nilearn.decoding import Decoder

decoder = Decoder(
    estimator="svc",
    mask=mask_path,
    standardize="zscore_sample",
    screening_percentile=...,   # TODO: pick a starting value (e.g. 5)
    scoring="accuracy",
    cv=cv,
)
```

**TODO:**
- [ ] Import `Decoder` and `LeaveOneGroupOut`.
- [ ] Create `cv = LeaveOneGroupOut()`.
- [ ] Choose a starting `screening_percentile` (5 is a reasonable first guess).
- [ ] Construct the `Decoder` with the mask and `estimator="svc"`.

In [19]:
from nilearn.decoding import Decoder
from sklearn.model_selection import LeaveOneGroupOut

cv = LeaveOneGroupOut()

# TODO: choose screening_percentile (start small, e.g. 5)
screening_percentile = 5
# TODO: build the Decoder
decoder = Decoder(
    estimator="svc",
    mask=mask_path,
    standardize="zscore_sample",
    screening_percentile=screening_percentile,
    scoring="accuracy",
    cv=cv,
)

---
## Section 8 — Fit the Decoder

`decoder.fit(imgs, y, groups=groups)` does a lot at once: it masks every image to
build `X`, then for **each CV fold** it z-scores, runs ANOVA screening on the
*training* trials, fits the SVM, and predicts the held-out run. Passing `groups`
is what tells `LeaveOneGroupOut` where the run boundaries are — forget it and the
split is meaningless.

```python
decoder.fit(imgs, y, groups=groups)
```

This will take a little while (it is loading and masking 32 volumes and fitting
two SVMs). That is expected.

**TODO:**
- [ ] Call `decoder.fit(imgs, y, groups=groups)`.
- [ ] After it returns, print `decoder.cv_scores_` to see per-fold accuracy.

In [20]:
# TODO: decoder.fit(imgs, y, groups=groups)
decoder.fit(imgs, y, groups=groups)
# TODO: print decoder.cv_scores_
print(decoder.cv_scores_)

C:\Users\jwhit\AppData\Local\Temp\ipykernel_25204\928408764.py:2: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)


{np.str_('neg_val'): [0.5625, 0.3125], np.str_('pos_val'): [0.5625, 0.3125]}


Section 8.5 I'm going to create a loop that fits many SVM models using different ANOVA cutoffs




In [23]:
cv = LeaveOneGroupOut()

# TODO: choose screening_percentile (start small, e.g. 5)
for screening_percentile in [1, 2, 3, 4]:
    # TODO: build the Decoder
    decoder = Decoder(
        estimator="svc",
        mask=mask_path,
        standardize="zscore_sample",
        screening_percentile=screening_percentile,
        scoring="accuracy",
        cv=cv,
    )
    # TODO: decoder.fit(imgs, y, groups=groups)
    decoder.fit(imgs, y, groups=groups)
    # TODO: print decoder.cv_scores_
    print(decoder.cv_scores_)

C:\Users\jwhit\AppData\Local\Temp\ipykernel_25204\2751255505.py:15: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)


{np.str_('neg_val'): [0.5, 0.375], np.str_('pos_val'): [0.5, 0.375]}


C:\Users\jwhit\AppData\Local\Temp\ipykernel_25204\2751255505.py:15: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)


{np.str_('neg_val'): [0.5, 0.3125], np.str_('pos_val'): [0.5, 0.3125]}


C:\Users\jwhit\AppData\Local\Temp\ipykernel_25204\2751255505.py:15: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)


{np.str_('neg_val'): [0.5, 0.3125], np.str_('pos_val'): [0.5, 0.3125]}


C:\Users\jwhit\AppData\Local\Temp\ipykernel_25204\2751255505.py:15: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)


{np.str_('neg_val'): [0.5625, 0.3125], np.str_('pos_val'): [0.5625, 0.3125]}


---
## Section 9 — Evaluate the Performance

`decoder.cv_scores_` is a dictionary keyed by class label; each value is a list
with one score per CV fold. To summarise, average the per-fold scores.

```python
import numpy as np

# cv_scores_ is keyed by class; for a balanced binary problem the per-fold
# accuracies are shared, so averaging any class's list gives the mean accuracy.
for label, scores in decoder.cv_scores_.items():
    print(label, np.mean(scores), scores)
```

**Interpret carefully:**

- Compare the mean to **chance = 50%**, not to 0.
- Two folds is a *very* small number of estimates — expect the two fold scores to
  bounce around. A single run-pair on one subject is noisy; do not over-read a
  few points above or below 50%.
- "Above chance" here is suggestive, not proof. Real studies repeat this across
  many subjects and test the *group* against chance.

**TODO:**
- [ ] Compute and print the mean cross-validated accuracy.
- [ ] State, in a markdown cell, how it compares to the 50% chance level.
- [ ] Note whether the two folds agree or disagree, and why that matters.

In [ ]:
import numpy as np

# TODO: average decoder.cv_scores_ and print mean accuracy per class / overall

*Your interpretation (mean accuracy vs 50% chance, fold agreement):*

> (write here)

---
## Section 10 — (Optional) Look at the Decoder Weight Map

A fitted `Decoder` exposes a weight image per class in `decoder.coef_img_` — a
brain map of how much each voxel contributed to the decision. Plotting it shows
*where* in the brain the discriminating pattern lives.

**A caution that matters scientifically:** SVM weights are **not** the same as a
univariate activation map. A voxel can have a large weight because it helps
*cancel noise* in another voxel, not because it is individually "active." Treat
the weight map as "what the model used," not "where valence is." (Nilearn's
tutorials discuss this; for formal interpretation, look up Haufe et al., 2014.)

**TODO (optional):**
- [ ] Plot `decoder.coef_img_[<class label>]` with `plot_stat_map`.
- [ ] Save it under `decoding/figures/`.

## Summary & what comes next

You built a full leak-free decoding pipeline for one binary question. Notebook 02
reuses *this exact pipeline* with a one-line change — the input table — to ask
the arousal question, so you can compare the two.

In [ ]:
# TODO (optional): plot and save the decoder weight map from decoder.coef_img_